In [ ]:
# Variogram model fitting and LOOCV — SSI without FI
# Fits spherical, exponential, and Gaussian models (isotropic and anisotropic)
# and ranks them by leave-one-out cross-validation. The final model adopted for
# mapping is the anisotropic exponential (see the paper and the mapping cell below).

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import curve_fit
from pykrige.ok import OrdinaryKriging

file_path = "Kriging_GN.xlsx"
target = "SSI without FI"

df = pd.read_excel(file_path, sheet_name="Sheet1").dropna(
    subset=["Easting", "Northing", target]).copy()
X = df["Easting"].to_numpy(float)
Y = df["Northing"].to_numpy(float)
V = df[target].to_numpy(float)

ANG_DEG = 160.0
RATIO_MINOR_MAJOR = 0.380
MIN_PAIRS = 100
nlags = 12

def experimental_variogram(x, y, z, maxlag, nlags):
    coords = np.c_[x, y]
    D = squareform(pdist(coords))
    iu = np.triu_indices(len(z), 1)
    h = D[iu]
    gamma_pairs = (0.5 * (z[:, None] - z[None, :]) ** 2)[iu]
    edges = np.linspace(0, maxlag, nlags + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    gamma, npairs = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (h > a) & (h <= b)
        npairs.append(m.sum())
        gamma.append(gamma_pairs[m].mean() if m.sum() > 0 else np.nan)
    return centers, np.array(gamma), np.array(npairs)

def rotate_and_scale(x, y, angle_deg, ratio):
    xc, yc = x - x.mean(), y - y.mean()
    theta = np.deg2rad(angle_deg)
    xr = xc * np.cos(theta) + yc * np.sin(theta)
    yr = -xc * np.sin(theta) + yc * np.cos(theta)
    return xr, yr / ratio

def spherical(h, nugget, psill, a):
    h = np.asarray(h)
    return nugget + psill * np.where(h < a, 1.5 * (h / a) - 0.5 * (h / a) ** 3, 1.0)

def exponential(h, nugget, psill, a):
    h = np.asarray(h)
    return nugget + psill * (1.0 - np.exp(-h / a))

def gaussian(h, nugget, psill, a):
    h = np.asarray(h)
    return nugget + psill * (1.0 - np.exp(-(h / a) ** 2))

models = {"spherical": spherical, "exponential": exponential, "gaussian": gaussian}

def fit_variogram_model(model_name, centers, gamma, npairs, maxlag):
    used = np.isfinite(gamma) & (npairs >= MIN_PAIRS)
    c, g, n = centers[used], gamma[used], npairs[used]
    if len(g) < 4:
        return None
    nug0 = max(0.0, np.nanmin(g))
    sill0 = max(np.nanpercentile(g, 80), nug0 + 1e-8)
    p0 = [nug0, max(sill0 - nug0, 1e-8), 0.4 * maxlag]
    f = models[model_name]
    popt, _ = curve_fit(f, c, g, p0=p0,
                        bounds=([0.0, 1e-10, 5000.0], [np.inf, np.inf, maxlag]),
                        sigma=1.0 / np.sqrt(n), maxfev=50000)
    nugget, psill, rng = popt
    return {"nugget": float(nugget), "psill": float(psill), "range": float(rng)}

def loocv_ok(coords, values, family, params, anisotropic=False):
    errors = []
    for i in range(len(values)):
        keep = np.ones(len(values), dtype=bool); keep[i] = False
        OK = OrdinaryKriging(
            coords[keep, 0], coords[keep, 1], values[keep],
            variogram_model=family,
            variogram_parameters={"psill": params["psill"], "range": params["range"], "nugget": params["nugget"]},
            anisotropy_angle=ANG_DEG if anisotropic else 0.0,
            anisotropy_scaling=RATIO_MINOR_MAJOR if anisotropic else 1.0,
            enable_plotting=False, verbose=False, pseudo_inv=True)
        z_pred, _ = OK.execute("points", np.array([coords[i, 0]]), np.array([coords[i, 1]]))
        errors.append(values[i] - float(np.asarray(z_pred).ravel()[0]))
    errors = np.array(errors)
    return {"RMSE": float(np.sqrt(np.mean(errors ** 2))),
            "MAE": float(np.mean(np.abs(errors))),
            "ME": float(np.mean(errors))}

coords = np.c_[X, Y].astype(float)
maxlag_iso = 0.5 * pdist(coords).max()
X_a, Y_a = rotate_and_scale(X, Y, ANG_DEG, RATIO_MINOR_MAJOR)
maxlag_ani = 0.5 * pdist(np.c_[X_a, Y_a]).max()

cent_iso, gam_iso, pairs_iso = experimental_variogram(X, Y, V, maxlag_iso, nlags)
cent_ani, gam_ani, pairs_ani = experimental_variogram(X_a, Y_a, V, maxlag_ani, nlags)

rows = []
for name in models:
    fi = fit_variogram_model(name, cent_iso, gam_iso, pairs_iso, maxlag_iso)
    fa = fit_variogram_model(name, cent_ani, gam_ani, pairs_ani, maxlag_ani)
    if fi:
        r = loocv_ok(coords, V, name, fi, anisotropic=False)
        rows.append({"Family": name, "Case": "Isotropic", **fi, **r})
    if fa:
        r = loocv_ok(coords, V, name, fa, anisotropic=True)
        rows.append({"Family": name, "Case": "Anisotropic", **fa, **r})

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
print("Variogram models ranked by LOOCV RMSE (SSI without FI):")
print(results.to_string(index=False))
print("\nFinal model adopted for mapping: anisotropic exponential")

In [ ]:
# Final kriging and normalized map — SSI without FI
# Uses the anisotropic exponential model selected above. The kriged surface is
# normalized to 0-1 for visualization and displayed with producing-well and
# active-permit overlays over the Powder River Basin.
# To run: place this notebook and all data files (Kriging_GN.xlsx, Overlaid_wells.xlsx,
# and the prbbndg shapefile set: .shp, .shx, .dbf, .prj) in the same folder.

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from pykrige.ok import OrdinaryKriging
from pyproj import Transformer

kriging_file = "Kriging_GN.xlsx"
overlay_file = "Overlaid_wells.xlsx"
boundary_file = "prbbndg.shp"
target = "SSI without FI"

df = pd.read_excel(kriging_file, sheet_name="Sheet1").dropna(
    subset=["Well Number", "Easting", "Northing", "Latitude", "Longitude", target]
).copy()
X = df["Easting"].to_numpy(float)
Y = df["Northing"].to_numpy(float)
V = df[target].to_numpy(float)

gdf_wells = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["Easting"], df["Northing"]), crs="EPSG:32613")

# Producing and active-permit overlays
loc_df = pd.read_excel(overlay_file, sheet_name="Producing and AP")
loc_df = loc_df.rename(columns={"Number": "number", "Latitude": "latitude", "Longitude": "longitude"})
producing_wells = loc_df[loc_df["number"] <= 116].copy()
ap_wells = loc_df[loc_df["number"] >= 117].copy()
producing_gdf = gpd.GeoDataFrame(
    producing_wells,
    geometry=gpd.points_from_xy(producing_wells["longitude"], producing_wells["latitude"]),
    crs="EPSG:4326").to_crs("EPSG:32613")
ap_gdf = gpd.GeoDataFrame(
    ap_wells,
    geometry=gpd.points_from_xy(ap_wells["longitude"], ap_wells["latitude"]),
    crs="EPSG:4326").to_crs("EPSG:32613")

# Basin boundary, buffered, with a 10% padded extent
prb = gpd.read_file(boundary_file).dissolve().to_crs("EPSG:32613")
prb_buffered = gpd.GeoDataFrame(geometry=prb.buffer(40000), crs="EPSG:32613")
minx, miny, maxx, maxy = prb_buffered.total_bounds
bx, by = (maxx - minx) * 0.10, (maxy - miny) * 0.10
minx -= bx; maxx += bx; miny -= by; maxy += by

grid_x = np.linspace(minx, maxx, 700)
grid_y = np.linspace(miny, maxy, 700)
grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)

# Final anisotropic exponential kriging
OK = OrdinaryKriging(
    X, Y, V,
    variogram_model="exponential",
    variogram_parameters={"psill": 0.000253, "range": 52083.871780, "nugget": 0.000373},
    anisotropy_angle=160.0, anisotropy_scaling=0.380,
    enable_plotting=False, verbose=False)
z, ss = OK.execute("grid", grid_x, grid_y)
ss = np.clip(ss, 0, None)


# Normalize to 0-1 for visualization
z_norm = (z - np.nanmin(z)) / (np.nanmax(z) - np.nanmin(z))

# Mask to the buffered basin
grid_points = gpd.GeoDataFrame(
    geometry=[Point(x, y) for x, y in zip(grid_xx.ravel(), grid_yy.ravel())], crs="EPSG:32613")
try:
    mask_geom = prb_buffered.union_all()
except Exception:
    mask_geom = prb_buffered.unary_union
mask = grid_points.within(mask_geom).to_numpy()

z_norm_masked = np.full_like(z_norm, np.nan)
ss_masked = np.full_like(ss, np.nan)
z_norm_masked.ravel()[mask] = z_norm.ravel()[mask]
ss_masked.ravel()[mask] = ss.ravel()[mask]

# 45 deg N WY-MT reference line
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32613", always_xy=True)
_, y_top = transformer.transform(-107, 45.0)
top_gdf = gpd.GeoDataFrame(geometry=[LineString([(minx, y_top), (maxx, y_top)])], crs="EPSG:32613")
to_latlon = Transformer.from_crs("EPSG:32613", "EPSG:4326", always_xy=True)

def set_latlon_ticks(ax):
    xticks = np.linspace(minx, maxx, 6)
    yticks = np.linspace(miny, maxy, 6)
    lon_ticks, _ = to_latlon.transform(xticks, [yticks[0]] * len(xticks))
    _, lat_ticks = to_latlon.transform([xticks[0]] * len(yticks), yticks)
    ax.set_xticks(xticks); ax.set_yticks(yticks)
    ax.set_xticklabels([f"{lon:.2f}" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.2f}" for lat in lat_ticks])
    ax.set_xlabel("Longitude (deg)"); ax.set_ylabel("Latitude (deg)")

def add_common_layers(ax, show_overlay=True):
    prb_buffered.boundary.plot(ax=ax, color="black", linewidth=1.2)
    top_gdf.plot(ax=ax, color="red", linestyle="--", linewidth=2, label="45N WY-MT boundary")
    gdf_wells.plot(ax=ax, color="black", markersize=25, marker="o", alpha=0.5, label="Kriging wells")
    if show_overlay:
        producing_gdf.plot(ax=ax, color="blue", markersize=25, marker="o", label="Producing wells")
        ap_gdf.plot(ax=ax, color="green", markersize=25, marker="^", label="Active permits")
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)
    set_latlon_ticks(ax)

# Normalized SSI map with kriging variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 12), dpi=150)

cf1 = ax1.contourf(grid_xx, grid_yy, z_norm_masked, levels=np.linspace(0, 1, 21), cmap="Spectral_r")
add_common_layers(ax1, show_overlay=True)
fig.colorbar(cf1, ax=ax1, shrink=0.82, aspect=24, pad=0.03).set_label("Normalized Average SSI without FI")
ax1.set_title("Normalized Average SSI without FI\nAnisotropic Exponential Kriging")
ax1.legend(loc="upper right", fontsize=9)
ax1.set_aspect("equal", adjustable="box")

cf2 = ax2.contourf(grid_xx, grid_yy, ss_masked,
                   levels=np.linspace(np.nanmin(ss_masked), np.nanmax(ss_masked), 20), cmap="plasma")
add_common_layers(ax2, show_overlay=False)
fig.colorbar(cf2, ax=ax2, shrink=0.82, aspect=24, pad=0.03).set_label("Kriging Variance")
ax2.set_title("Kriging Variance\nNormalized Average SSI without FI")
ax2.legend(loc="upper right", fontsize=9)
ax2.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.show()